In [2]:
import torch as tr
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken as ttk
import numpy as np
import matplotlib.pyplot as pl
import pandas as pd
import sys
import os

# Get the path of the directory you want to import from
module_dir = [os.path.relpath('GPT_architecture'), os.path.relpath('data')]

# Add the directory to the system path
for direc in module_dir:
    if direc not in sys.path:
        sys.path.append(direc)

print(sys.path)
# Now you can import the module as if it were in the current directory

import GPT_modules as gpt

GPT_CONFIG_124M = {
"vocab_size": 50257, # Vocabulary size
"context_length": 256, # Context length
"emb_dim": 768, # Embedding dimension
"n_heads": 12, # Number of attention heads
"n_layers": 12, # Number of layers
"drop_rate": 0.1, # Dropout rate
"qkv_bias": True # Query-Key-Value bias
}

tokenizer = ttk.get_encoding("gpt2")
# gpt_model = gpt.GPTModule(GPT_CONFIG_124M)
# gpt_model.eval()


['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', 'GPT_architecture', 'data']


### Downloading the dataset

In [4]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
            with open(file_path, "w", encoding="utf-8") as file:
                file.write(text_data)
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

file_path = "data/instruction-data.json"
url = (
"https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
"/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


### Data formatting

In [5]:
def format_input(input_data):
    prefix_instruction = (f"Below is an instruction that defines the task. " 
                          f"Write an appropriate response completing the task.\n\n"
                          f"### Instruction: {input_data['instruction']} \n\n")

    input_prompt = (f"### Input: {input_data['input']} \n\n" if input_data['input'] else '')
    return prefix_instruction + input_prompt
frt_string = format_input(data[999])
response = f"### Output: {data[999]['output']}"
frt_data = frt_string + response
print(frt_data)

Below is an instruction that defines the task. Write an appropriate response completing the task.

### Instruction: What is an antonym of 'complicated'? 

### Output: An antonym of 'complicated' is 'simple'.


In [6]:
tokenizer = ttk.get_encoding('gpt2')
tokenizer.encode('<|endoftext|>',allowed_special={'<|endoftext|>'})

[50256]

In [7]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer
        self.encoded_data = []
        for text in data:
            formatted_input = format_input(text)
            response = (f"### Input: {text['input']} \n\n" if text['input'] else '')
            formatted_text = formatted_input + response
            self.encoded_data.append(tokenizer.encode(formatted_text))
        
    def __getitem__(self,index):
        return self.encoded_data[index]
    def __len__(self):
        return len(self.data)

instruct_dataset = InstructionDataset(data, tokenizer)

### Customizing the collate function to handle padding

In [23]:
def custom_collate_draft1(batch, pad_token_id=50256, device='cpu'):
    batch_max_len = max(len(data) for data in batch)

    encoded_list = []
    for data in batch:
        new_data = data.copy()
        padded = new_data + [pad_token_id]*(batch_max_len - len(new_data))
        inpt = tr.tensor(padded)
        encoded_list.append(inpt)
    # print(encoded_list)
    input_tensor = tr.stack(encoded_list).to(device)

    return input_tensor

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1, inputs_2, inputs_3)
print(custom_collate_draft1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [7]:
max(len(data)+1 for data in instruct_dataset.encoded_data) 

79

In [24]:
def custom_collate_draft2(batch, pad_token_id=50256, device='cpu'):
    
    batch_max_len = max(len(data)+1 for data in batch)

    encoded_list, target_list = [], []
    for data in batch:
        new_data = data.copy()
        new_data += [pad_token_id]

        padded = new_data + [pad_token_id]*(batch_max_len - len(new_data))
        inpt = tr.tensor(padded[:-1])
        trgt = tr.tensor(padded[1:])
        encoded_list.append(inpt)
        target_list.append(trgt)

    # print(encoded_list)
    input_tensor = tr.stack(encoded_list).to(device)
    target_tensor = tr.stack(target_list).to(device)

    return input_tensor, target_tensor

inpt, trgt = custom_collate_draft2(batch)
inpt, trgt

(tensor([[    0,     1,     2,     3,     4],
         [    5,     6, 50256, 50256, 50256],
         [    7,     8,     9, 50256, 50256]]),
 tensor([[    1,     2,     3,     4, 50256],
         [    6, 50256, 50256, 50256, 50256],
         [    8,     9, 50256, 50256, 50256]]))

In [25]:
(trgt[1] == -100)[:]
# trgt
mask = trgt[1] == -100
indices = tr.nonzero(mask).squeeze()
indices
indices, indices.numel()


(tensor([], dtype=torch.int64), 0)

In [8]:
def custom_collate_func(batch, pad_token_id=50256, ignore_index=-100, allowed_max_len = None, device='cpu'):
    
    ## max len is added by 1 
    ## since a padded token is added to data initially
    batch_max_len = max(len(data)+1 for data in batch)

    encoded_list, target_list = [], []
    for data in batch:
        new_data = data.copy()
        ## this token is added since the target will be shifted by 1
        new_data += [pad_token_id]

        padded = new_data + [pad_token_id]*(batch_max_len - len(new_data))
        inpt = tr.tensor(padded[:-1])
        trgt = tr.tensor(padded[1:])

        ## all indices where pad is added
        mask_ind = trgt == pad_token_id
        ## need to keep the last pad and change all else to -100
        if (batch_max_len - len(new_data)):
            ## if padding is added, change them to -100
            mask_ind[len(new_data)-2] = False
            trgt[mask_ind] = ignore_index

        if allowed_max_len is not None:
            inpt = inpt[:allowed_max_len]
            trgt = trgt[:allowed_max_len]

        encoded_list.append(inpt)
        target_list.append(trgt)

    # print(encoded_list)
    input_tensor = tr.stack(encoded_list).to(device)
    target_tensor = tr.stack(target_list).to(device)

    return input_tensor, target_tensor

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [6]
inputs_3 = [7, 8, 9]
batch = (inputs_1, inputs_2, inputs_3)
# print(custom_collate_draft1(batch))
inpt, trgt = custom_collate_func(batch)
inpt, trgt

(tensor([[    0,     1,     2,     3,     4],
         [    6, 50256, 50256, 50256, 50256],
         [    7,     8,     9, 50256, 50256]]),
 tensor([[    1,     2,     3,     4, 50256],
         [50256,  -100,  -100,  -100,  -100],
         [    8,     9, 50256,  -100,  -100]]))

In [9]:
from functools import partial

# device = tr.device("cuda" if tr.cuda.is_available() else "cpu")

if tr.cuda.is_available():
    device = tr.device("cuda")
elif tr.mps.is_available():
    device = tr.device("mps")
else:
    device = tr.device("cpu")

customized_collate_func = partial(custom_collate_func,device=device,allowed_max_len=1024)

### Using the collate function in DataLoader

In [ ]:
num_workers = 0
batch_size = 8
tr.manual_seed(1)

train_ind = int(len(data)*0.85)
test_ind = train_ind + int(len(data)*0.1)

train_data = data[:train_ind]
test_data = data[train_ind:test_ind]
valid_data = data[test_ind:]

train_dataset = InstructionDataset(train_data, tokenizer)
test_dataset = InstructionDataset(test_data, tokenizer)
valid_dataset = InstructionDataset(valid_data, tokenizer)

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = batch_size,
    collate_fn = customized_collate_func,
    shuffle = True, 
    drop_last = True, 
    num_workers = num_workers
)

test_dataloader = DataLoader(
    test_dataset, 
    batch_size = batch_size,
    collate_fn = customized_collate_func,
    shuffle = True, 
    drop_last = True, 
    num_workers = num_workers
)

valid_dataloader = DataLoader(
    valid_dataset, 
    batch_size = batch_size,
    collate_fn = customized_collate_func,
    shuffle = True, 
    drop_last = Tr`ue, 
    num_workers = num_workers
)

In [ ]:
for eg in train_dataloader:
    inpt, trgt = eg
    print(inpt.shape, trgt.shape)

### Connecting to GCP

In [ ]:
# from google.colab import auth

# auth.authenticate_user()
# print('User authenticated.')



AuthorizationError: Error fetching credentials

### Downloading the GPT model weights for 355 M parameters

In [9]:
from gpt_download import download_and_load_gpt2
from GPT_modules import GPTModule
from auxiliary_functions import load_weights_into_model

BASE_CONFIG = {"vocab_size": 50257,
               "context_length": 1024,
               "drop_rate": 0.0,
               "qkv_bias": True}

model_configs = {"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
                 "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
                 "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
                 "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
model_size=model_size,
models_dir="gpt2"
)


File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe


In [12]:
from GPT_modules import GPTModule

model = GPTModule(BASE_CONFIG)
load_weights_into_model(model, params)
model.eval()

GPTModule(
  (Token_Embedding): Embedding(50257, 1024)
  (Position_Embedding): Embedding(1024, 1024)
  (Dropout_bef_Transform): Dropout(p=0.0, inplace=False)
  (Transformer_block): Sequential(
    (0): Transformer(
      (Layer_norm1): LayerNormalization()
      (Layer_norm2): LayerNormalization()
      (Multihead_attn): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (FFN): FFN(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (Dropout): Dropout(p=0.0, inplace=False)
    )
    (1

In [58]:
model = GPTModule(BASE_CONFIG)
load_weights_into_model(model, params)
model.eval()

GPTModule(
  (Token_Embedding): Embedding(50257, 1024)
  (Position_Embedding): Embedding(1024, 1024)
  (Dropout_bef_Transform): Dropout(p=0.0, inplace=False)
  (Transformer_block): Sequential(
    (0): Transformer(
      (Layer_norm1): LayerNormalization()
      (Layer_norm2): LayerNormalization()
      (Multihead_attn): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (FFN): FFN(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (Dropout): Dropout(p=0.0, inplace=False)
    )
    (1

In [ ]:
from auxiliary_functions import generate

# if tr.cuda.is_available():
#     device = tr.device("cuda")
# elif tr.mps.is_available():
#     device = tr.device("mps")
# else:
#     device = tr.device("cpu")
    
# input_sentence = "Every thing"
# # top_k = 3
# device = "cpu"
input_sentence = format_input(valid_data[0])
model.to(device)
# print(device)
# training_args = tr.TrainingArguments(use_mps_device=True, no_cuda=True)
# model(tokenizer.encode(input_sentence))
# model(tr.tensor(tokenizer.encode(input_sentence), device=device))
text_gen = generate(input_sentence, tokenizer, model, device, GPT_CONFIG_124M["context_length"], temperature=2, max_length_text=10, top_k=3)
# print(f"INPUT: {input_sentence}")
# print(f"OUTPUT: {text_gen}")



In [17]:
input_sentence = "Dhoomketu a Gujarati author"
text_gen = generate(input_sentence, tokenizer, model, GPT_CONFIG_124M["context_length"], temperature=2, max_length_text=20, top_k=3)
print(f"INPUT: {input_sentence}")
print(f"OUTPUT: {text_gen}")


INPUT: Dhoomketu a Gujarati author
OUTPUT: Dhoomketu a Gujarati author who was the first to use the word 'hindus', and was also the first person to


In [56]:

# def calc_accuracy_loader(data_loader, model, num_batches=None):
#     model.eval()
#     correct_pred, num_examples = 0, 0

#     if num_batches == None:
#         num_batches = len(data_loader)
#     else:
#         num_batches = min(num_batches,len(data_loader))

#     with tr.no_grad():
#         correct_pred = 0
#         for inp, label in data_loader:
#             out = model(inp)
#             pred_labels_batch = tr.argmax(out[:,-1,:],dim=1)
#             correct_pred += (pred_labels_batch == label).sum().item()
#             num_examples += len(label)
    
#     return correct_pred/num_examples


def calc_loss_batch(input, target, device, model):

    input = input.to(device)
    target = target.to(device)
    print(input.device, next(model.parameters()).device)
    out_flat = model(input).flatten(0,1)
    
    tar_flat = target.flatten()
    # print(out_flat.shape,tar_flat.shape)
    return nn.functional.cross_entropy(out_flat, tar_flat, reduction='mean')

def calc_loss_loader(data_loader, model, device, num_batches=None):

    loss = 0

    if num_batches == None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches,len(data_loader))

    for count, (inp, label) in enumerate(data_loader):
        if count < num_batches:
            loss += calc_loss_batch(inp, label, device, model).item()
        else:
            break
    
    return loss/num_batches

def eval_batch_loss(data_loader, model):
    loss = 0
    model.eval()

    # if num_batches == None:
    #     num_batches = len(data_loader)
    # else:
    #     num_batches = min(num_batches,len(data_loader))

    num_batches = len(data_loader)
    for inp, trgt in data_loader:
        output = model(inp)
        out_flat = output.flatten(0,1)
        tar_flat = trgt.flatten()
        loss += nn.functional.cross_entropy(out_flat, tar_flat)
        
    return loss/num_batches

def eval_loss(train_loader, validation_loader, model):

    train_loss = eval_batch_loss(train_loader, model)
    valid_loss = eval_batch_loss(validation_loader, model)

    return train_loss, valid_loss


def train_model_generic(model, n_epoch, optimizer, device, train_loader, validation_loader):
    loss_epoch = []
    train_losses = []
    valid_losses = []
    for epoch in range(n_epoch):
        model.train()
        loss_batches = 0
        for i, batch_data in enumerate(train_loader):
            inp, target = batch_data
            optimizer.zero_grad()
            loss = calc_loss_batch(inp, target, device, model) 
            loss.backward()
            optimizer.step()
            loss_batches += loss
            # if i%10 == 0:
            #     print(f"Losses in Epoch: {epoch}, batch no.: {i}")
            #     train_loss, valid_loss = eval_loss(train_loader, validation_loader, model)
            #     train_losses.append(train_loss)
            #     valid_losses.append(valid_loss)
            #     print(f"Train loss: {train_loss:0.3f}")
            #     print(f"Validation loss: {valid_loss:0.3f}")

        
        loss_epoch.append(loss_batches)

    return loss_epoch, train_losses, valid_losses


In [59]:
# from auxiliary_functions import (calc_loss_loader, train_model_generic)
import time

# model.load_state_dict(tr.load("instruction_model.pth"))


model.to(device)
tr.manual_seed(1)

with tr.no_grad():
    train_loss = calc_loss_loader(train_dataloader, model, device, num_batches=5)
    valid_loss = calc_loss_loader(valid_dataloader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", valid_loss)

RuntimeError: MPS backend out of memory (MPS allocated: 9.05 GiB, other allocations: 912.00 KiB, max allowed: 9.07 GiB). Tried to allocate 16.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [41]:
import time 

start_time = time.time()
# loss_epoch
n_epoch = 1
optimizer = tr.optim.AdamW(model.parameters(),lr=5e-5, weight_decay=0.1)

loss_epoch, train_losses, valid_losses = train_model_generic(model, n_epoch, optimizer, device, train_dataloader, valid_dataloader)

end_time = (time.time() - start_time)/60

print(f"Time for training:{end_time:0.3f}")

mps:0 mps:0


RuntimeError: Placeholder storage has not been allocated on MPS device!

In [21]:
tr.save(model.state_dict(), "instruction_model.pth")

AttributeError: 'GPTModule' object has no attribute 'weight'

In [27]:
# tr.load("instruction_model.pth")


if tr.cuda.is_available():
    device = tr.device("cuda")
elif tr.mps.is_available():
    device = tr.device("mps")
else:
    device = tr.device("cpu")
model.to(device)
next(model.parameters()).device

device(type='mps', index=0)

In [70]:
# from datasets import load_dataset
import datasets
# dir(load_dataset)
# type(datasets)
datasets

dir(datasets)

['Array2D',
 'Array3D',
 'Array4D',
 'Array5D',
 'ArrowBasedBuilder',
 'Audio',
 'BuilderConfig',
 'ClassLabel',
 'Column',
 'Dataset',
 'DatasetBuilder',
 'DatasetDict',
 'DatasetInfo',
 'DownloadConfig',
 'DownloadManager',
 'DownloadMode',
 'Features',
 'GeneratorBasedBuilder',
 'Image',
 'IterableColumn',
 'IterableDataset',
 'IterableDatasetDict',
 'LargeList',
 'List',
 'NamedSplit',
 'NamedSplitAll',
 'Nifti',
 'Pdf',
 'ReadInstruction',
 'Sequence',
 'Split',
 'SplitBase',
 'SplitDict',
 'SplitGenerator',
 'SplitInfo',
 'StreamingDownloadManager',
 'SubSplitInfo',
 'Translation',
 'TranslationVariableLanguages',
 'Value',
 'VerificationMode',
 'Version',
 'Video',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'are_progress_bars_disabled',
 'arrow_dataset',
 'arrow_reader',
 'arrow_writer',
 'builder',
 'combine',
 'concatenate_datasets',
 'config',
 'data_files',
 'dataset_dict',
 